In [1]:
import sys
import os

# Set up the path to allow importing from the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import the necessary tests for Section 4
from tests.test import (
    test_number_of_bits,
    test_bit_names,
    test_total_input_bits,
    test_encode_cells,
    test_encode_sample,
    test_decode_cells,
    test_invalid_binary_pattern,
    test_bit_ownership
)

print("Setup complete. Section 4 tests imported successfully.")

Setup complete. Section 4 tests imported successfully.


### 4.1 Bit Allocation and Variable Naming

> *"To represent the discretized cells in a Boolean domain, each feature requires a specific number of boolean variables (bits). If a feature has $K$ cells, we allocate $B = \lceil \log_2(K) \rceil$ bits."*

**Explanation:** In order to evaluate the random forest using Binary Decision Diagrams (BDDs) or other boolean logic systems, we must convert the integer cell indices into binary representations. For example, a feature with 4 cells needs 2 bits (states 00, 01, 10, 11). Each bit is assigned a specific standardized name (e.g., `x0_0`, `x0_1` for the first and second bit of feature 0) so we can track them across the entire dataset.

**Test:** 
- `test_number_of_bits()` ensures the correct formula is applied to calculate the bits required per feature.
- `test_bit_names()` verifies the correct generation of standard variable names.
- `test_total_input_bits()` checks the total boolean dimensionality of the input space.

In [2]:
test_number_of_bits()
test_bit_names()
test_total_input_bits()

[PASS] feature 0 uses 2 bits and feature 1 uses 1 bit
[PASS] bit names are correct
[PASS] total number of input bits is 3


### 4.2 Binary Encoding of Samples

> *"Each integer cell index is translated into its standard binary representation using the allocated bits. A full continuous sample can thus be mapped directly to a boolean assignment."*

**Explanation:** This is the forward transformation. Once we know which integer cell a real value falls into (Section 3), we convert that integer into a binary sequence (True/False boolean assignments). For example, cell `2` with 2 bits becomes binary `10`, which translates to `{bit_0: False, bit_1: True}`. We can do this from pre-calculated cell arrays, or directly from the original raw float values.

**Test:** 
- `test_encode_cells()` validates that integer arrays are mapped to the correct boolean dictionaries.
- `test_encode_sample()` validates the end-to-end pipeline: raw float sample $\rightarrow$ cell index $\rightarrow$ boolean assignment.

In [3]:
test_encode_cells()
test_encode_sample()

[PASS] encode_cells([0, 0]) is correct
[PASS] encode_cells([1, 0]) is correct
[PASS] encode_cells([2, 0]) is correct
[PASS] encode_cells([3, 1]) is correct
[PASS] encode_sample([7.0, 10.1]) is correct


### 4.3 Decoding and the Aliasing of Invalid Patterns

> *"When decoding a binary assignment back to cell indices, we must handle invalid patterns. If a feature has 3 cells (indices 0, 1, 2), it requires 2 bits. However, 2 bits can represent 4 states (0, 1, 2, 3). The invalid binary pattern representing '3' must alias to the last valid cell ('2')."*

**Explanation:** Because the number of cells is not always a perfect power of 2, the boolean space is sometimes larger than the discrete feature space. If a boolean formula eventually evaluates an "impossible" binary state, we must map it safely back to reality. The rule is simple: any binary integer greater than the maximum valid cell index is forced (aliased) to the maximum valid cell index. 

**Test:** 
- `test_decode_cells()` verifies the correct reverse mapping from boolean dictionaries to cell indices for normal cases.
- `test_invalid_binary_pattern()` explicitly tests the aliasing rule by forcing an out-of-bounds binary state and asserting that it correctly clamps to the highest valid cell index.

In [4]:
test_decode_cells()
test_invalid_binary_pattern()

[PASS] decode_cells({'x0_0': False, 'x0_1': False, 'x1_0': False}) is correct
[PASS] decode_cells({'x0_0': True, 'x0_1': False, 'x1_0': False}) is correct
[PASS] decode_cells({'x0_0': False, 'x0_1': True, 'x1_0': False}) is correct
[PASS] decode_cells({'x0_0': True, 'x0_1': True, 'x1_0': True}) is correct
[PASS] invalid binary pattern 3 aliases to the last cell 2


### 4.4 Bit Ownership Mapping

> *"For reverse engineering and analytical purposes, it is necessary to maintain a mapping from any given boolean variable back to its source feature and significance position."*

**Explanation:** When working with boolean formulas, we often look at a variable (like `x1_0`) and need to know programmatically what it represents. The `bit_owner()` metadata provides a fast dictionary mapping each boolean variable name string back to a tuple of `(feature_index, bit_position)`. 

**Test:** `test_bit_ownership()` checks that the generated dictionary perfectly matches the expected structure.

In [5]:
test_bit_ownership()

[PASS] every bit is associated with the correct feature and position
